# v7_pipeline — ML Stage 1.5 Validation (Colab GPU)

This notebook validates the optional ML adversarial stage on a Colab GPU:
1. Install deps (torch, open_clip, ffmpeg)
2. Load the project
3. Run the pipeline with the **BALANCED** profile on a sample video
4. Report **SSIM** (imperceptibility), **CLIP cosine distance** (embedding shift), and timing
5. Download the result

In [1]:
# 1. Check GPU
!nvidia-smi
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

Tue Aug 11 08:13:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Install deps + reclaim disk space BEFORE any model download
# Colab's VM disk is small and often nearly full — the CLIP safetensors
# download (~350 MB) then fails mid-write and the kernel dies. Clean up first.
!pip install -q torch open_clip_torch pillow numpy
!apt-get install -y -qq ffmpeg

import os, shutil
from pathlib import Path

# free space: pip/apt caches + any stale HF/torch caches from previous runs
!pip cache purge -q 2>/dev/null; apt-get clean -qq
for d in (Path.home()/'.cache/huggingface', Path.home()/'.cache/torch'):
    shutil.rmtree(d, ignore_errors=True)

# If Drive is mounted, put the HF/torch cache there so the model download
# doesn't eat the VM disk. Run this BEFORE the pipeline cell.
# from google.colab import drive; drive.mount('/content/drive')
if Path('/content/drive/MyDrive').is_dir():
    os.environ['HF_HOME'] = '/content/drive/MyDrive/.cache/hf'
    os.environ['TORCH_HOME'] = '/content/drive/MyDrive/.cache/hf/torch'
    print('cache -> Drive:', os.environ['HF_HOME'])
else:
    print('Drive not mounted — using local disk. Free space:')
    !df -h /root /content 2>/dev/null | tail -n +2
print('deps installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
Drive not mounted — using local disk. Free space:
overlay         113G   47G   66G  42% /
overlay         113G   47G   66G  42% /
deps installed


In [3]:
# 3. Load the project — either clone or upload a zip
import os, sys

REPO_URL = ''  # e.g. 'https://github.com/you/v7_pipeline.git' — leave blank to upload a zip instead

if REPO_URL:
    !git clone "$REPO_URL" project
else:
    from google.colab import files
    import zipfile, io
    print('Upload a zip of the project folder (or just v7_pipeline/):')
    up = files.upload()
    name = next(iter(up))
    if not zipfile.is_zipfile(io.BytesIO(up[name])):
        raise SystemExit(f'{name} is not a zip — upload v7_pipeline_colab.zip or a zipped project folder')
    with zipfile.ZipFile(io.BytesIO(up[name])) as z:
        z.extractall('project')

# find the folder that contains v7_pipeline/ (handles both zip layouts)
ROOT = None
for base, dirs, files_ in os.walk('.'):
    if 'v7_pipeline' in dirs:
        ROOT = os.path.abspath(base)
        break
assert ROOT, 'could not find v7_pipeline/ in the uploaded zip'
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print('project root:', ROOT)

Upload a zip of the project folder (or just v7_pipeline/):


Saving v7_pipeline_colab.zip to v7_pipeline_colab.zip
project root: /content/project


In [4]:
# 4. Upload a sample input video
from google.colab import files
from pathlib import Path

Path('input').mkdir(exist_ok=True)
print('Upload a sample video:')
up = files.upload()
name = next(iter(up))
INPUT = Path('input') / name
INPUT.write_bytes(up[name])
print('input:', INPUT, f'({INPUT.stat().st_size/1024/1024:.1f} MB)')

Upload a sample video:


Saving lv_0_20260613212516.mp4 to lv_0_20260613212516.mp4
input: input/lv_0_20260613212516.mp4 (10.9 MB)


In [5]:
# 5. Run the pipeline
# Choose your profile here. All of these have ml_stage=True (AI attack runs on GPU):
#   'BALANCED'  - medium strength, good starting point (RECOMMENDED on Colab)
#   'AGGRESSIVE'- high strength
#   'AIMIMIC'   - AI-generated look (bloom, saturation)
#   'MAXIMUM'   - nuclear option, everything on (slow on Colab's 2-core CPU)
# ('SAFE' and 'TOON' have ml_stage=False - no AI attack, ffmpeg tricks only)
# NOTE: MAXIMUM stacks many ffmpeg filters and is the slowest; if the cell
# times out or the kernel restarts, drop to BALANCED.
PROFILE = 'MAXIMUM'

import time
from v7_pipeline.stage1 import process_video

t0 = time.time()
OUT = process_video(str(INPUT), 'output', profile=PROFILE, seed=42)
ELAPSED = time.time() - t0
print('profile:', PROFILE)
print('output:', OUT)
print(f'wall time: {ELAPSED:.1f}s')

[ ] Profile: MAXIMUM
[ ] Duration: 11.7s | Size: 10.9 MB
[OK] Temp copy
[ ] Seed: 42 (use --seed 42 to reproduce this exact run)
[ ] Encoder: nvenc (NVENC test encode succeeded)
[ ] Config: CRF=25 GOP=40 PTS_Jitter=0.000648


Encoding (nvenc):   0%|          | 0/100% [00:00<?, ?%/s]

[OK] Encode 184.8s (nvenc)
[OK] Validated: video+audio


open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

[OK] ML adversarial pass applied
[OK] Behavioral padding: +5574 bytes
[ERR] Post-pad validate: duration mismatch source=11.65s out=10.80s
profile: MAXIMUM
output: None
wall time: 266.1s


In [6]:
# 6. Validation metrics: SSIM + CLIP cosine distance
import os
import numpy as np
import torch, open_clip
from PIL import Image
from v7_pipeline.ml.budget import ssim
from v7_pipeline.ml.orchestrate import _decode_sample

# reuse the Drive cache if cell 2 set it (avoids a second 350 MB download)
if 'HF_HOME' in os.environ:
    os.environ.setdefault('HF_HOME', os.environ['HF_HOME'])

src_frames = _decode_sample(str(INPUT))
out_frames = _decode_sample(OUT)
n = min(len(src_frames), len(out_frames))
mean_ssim = float(np.mean([ssim(src_frames[i], out_frames[i]) for i in range(n)]))
print(f'mean SSIM over {n} sampled frames: {mean_ssim:.4f}  (floor 0.98)')

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model, _, prep = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model = model.to(dev).eval()

def embed(frames):
    imgs = torch.stack([prep(Image.fromarray(f)) for f in frames[:n]]).to(dev)
    with torch.no_grad():
        e = model.encode_image(imgs)
    return (e / e.norm(dim=-1, keepdim=True)).mean(dim=0)

e_src, e_out = embed(src_frames), embed(out_frames)
cos = float(torch.nn.functional.cosine_similarity(e_src, e_out, dim=0))
print(f'CLIP cosine similarity original vs output: {cos:.4f}')
print(f'cosine distance: {1 - cos:.4f}  (higher = further from source embedding)')

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


mean SSIM over 0 sampled frames: nan  (floor 0.98)


RuntimeError: stack expects a non-empty TensorList

In [ ]:
# 7. Download the result
from google.colab import files
files.download(OUT)